In [ ]:
import os
import json
import random

def calculate_penalty(proposed_group1, proposed_group2, history):
    penalty = 0
    # history는 최신순으로 정렬된 과거 모임 기록 리스트
    for idx, past_meeting in enumerate(history):
        # 최근일수록 높은 가중치 부여 (1, 0.5, 0.33, 0.25 ...)
        weight = 1.0 / (idx + 1) 
        past_groups = past_meeting['groups']
        
        # 새로 짠 두 그룹에 대해 각각 검사
        for proposed_group in [proposed_group1, proposed_group2]:
            for i in range(len(proposed_group)):
                for j in range(i + 1, len(proposed_group)):
                    p1, p2 = proposed_group[i], proposed_group[j]
                    
                    # 두 사람이 과거 모임에서 같은 조였는지 확인
                    for pg in past_groups:
                        if p1 in pg and p2 in pg:
                            penalty += weight
                            break
    return penalty

def divide_groups(attendees, meeting_date_str, folder_path="history"):
    # 폴더가 없으면 생성
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        
    # 과거 기록 읽어오기
    history = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            filepath = os.path.join(folder_path, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                history.append({
                    'date': filename.replace('.json', ''),
                    'groups': [data.get('group1', []), data.get('group2', [])]
                })
                
    # 날짜를 기준으로 최신순 정렬
    history.sort(key=lambda x: x['date'], reverse=True)
    
    # 인원 분배 (반으로 나누기, 홀수면 한 쪽이 1명 더 많아짐)
    n = len(attendees)
    size1 = n // 2
    
    best_division = None
    min_penalty = float('inf')
    
    target_person = "오정환"
    has_target = target_person in attendees
    
    # 섞기 위한 풀(pool) 생성
    pool = attendees.copy()
    if has_target:
        pool.remove(target_person)
    
    # 1000번 무작위로 섞어보며 가장 겹침이 적은(벌점이 낮은) 조합 찾기
    for _ in range(1000):
        random.shuffle(pool)
        
        if has_target:
            g1 = pool[:size1]
            # 오정환을 무조건 2조에 포함
            g2 = pool[size1:] + [target_person]
        else:
            g1 = pool[:size1]
            g2 = pool[size1:]
        
        penalty = calculate_penalty(g1, g2, history)
        
        # 벌점이 더 낮은 조합을 발견하면 업데이트
        if penalty < min_penalty:
            min_penalty = penalty
            best_division = (list(g1), list(g2))
            
        # 벌점이 0이면 완벽히 안 겹친 것이므로 바로 종료
        if penalty == 0:
            break
            
    final_g1, final_g2 = best_division
    
    # 파일 이름에 / 대신 - 사용 (OS 제한 때문)
    safe_date_str = meeting_date_str.replace('/', '-')
    filepath = os.path.join(folder_path, f"{safe_date_str}.json")
    
    # 결과 저장
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump({"group1": final_g1, "group2": final_g2}, f, ensure_ascii=False, indent=2)
        
    return final_g1, final_g2

# --- 실행 부분 ---
# 매주 모임 때마다 이 부분의 명단과 날짜만 바꿔서 실행
today_attendees = ["구교진", "김택민", "이동건", "박찬하", "현지용", "장하연"]


today_date = "2026/06/21"

group1, group2 = divide_groups(today_attendees, today_date, './history')

print(f"[{today_date} 목장 모임 조 편성 결과]")
print(f"1조 ({len(group1)}명): {', '.join(group1)}")
print(f"2조 ({len(group2)}명): {', '.join(group2)}")

[2026/06/21 목장 모임 조 편성 결과]
1조 (3명): 장하연, 김택민, 구교진
2조 (3명): 이동건, 현지용, 박찬하
